# Satellite Pattern of Life Identification (SPLID) Dataset exploration


In [1]:
from pathlib import Path 
import os
def find_root(marker: str = "pyproject.toml"):
    p = Path.cwd().resolve()
    for parent in (p, *p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"{marker} introuvable en remontant depuis {p}")
ROOT = find_root()

DATA_DIR  = os.path.join(ROOT, 'data', 'raw', 'splid_dataset')
print(DATA_DIR)
## Fichier splid-dataset organisé en /test / training: 
## fichiers csv 

/home/echevrolat/Documents/dev/ssa-analysis/data/raw/splid_dataset


In [2]:
import pandas as pd
import os 

data_dir = os.path.join(DATA_DIR, 'training')
labels_dir = os.path.join(DATA_DIR, 'train_label.csv')



## Testing data pipeline from raw .csv to WindoDataset (pytorch)

In [3]:
ex_sat_path= os.path.join(data_dir, '00001.csv')
sat_ex = pd.read_csv(ex_sat_path)

sat_ex.columns ## accéder aux colonnes du csv
sat_ex['Timestamp'] ## accéder à une colonne en particulier 
sat_ex.iloc[1] ## accéder à la première ligne du csv 

print(sat_ex.iloc[0])


Timestamp                      2022-09-01 00:00:00.000000Z
Eccentricity                                      0.000202
Semimajor Axis (m)                         42165366.782587
Inclination (deg)                                 0.139822
RAAN (deg)                                       94.427336
Argument of Periapsis (deg)                      52.733092
True Anomaly (deg)                               277.81098
Latitude (deg)                                    -0.01446
Longitude (deg)                                  85.119506
Altitude (m)                               35786071.639079
X (m)                                      17838370.015089
Y (m)                                      38204848.972088
Z (m)                                        -50599.110117
Vx (m/s)                                      -2786.228658
Vy (m/s)                                       1300.258746
Vz (m/s)                                          6.534142
Name: 0, dtype: object


In [4]:
from ml.datahandler import load_splid_objects
from ml.dataset import make_loaders

objs, labels = load_splid_objects(data_dir, labels_dir)



In [5]:
train_dl, val_dl , meta = make_loaders(objs, labels)


In [6]:
x,y=next(iter(train_dl))
print(x.shape, y.shape)
 ## les 97 valeurs (normalisées) du paramètre k 
 # sur le premier vecteurs time-series de 00001.csv 
print(x[0][0].shape)


torch.Size([256, 9, 97]) torch.Size([256, 2])
torch.Size([97])


### Flash test pour la boucle d'entrainement 


In [15]:
x, y = next(iter(train_dl))

import torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = x.to(device)
y = y.to(device)
from ml.model import NaiveBaseLine
model = NaiveBaseLine(9,97,2).to(device)

optimizer = torch.optim.AdamW(model.parameters())
loss_fn = torch.nn.BCEWithLogitsLoss()
running_loss = 0.0
batch_size = 256 
for k in range (1, 200):

        optimizer.zero_grad()

        pred = model(x)

        loss = loss_fn(pred, y)

        running_loss+= loss.item() * y.size(0)

        loss.backward()

        optimizer.step()


        if k % 20 == 0 : 
                print(f"loss at iteration {k}: {running_loss / (k*batch_size)}")


loss at iteration 20: 0.6575116127729416
loss at iteration 40: 0.6035429999232292
loss at iteration 60: 0.5661478772759437
loss at iteration 80: 0.5364944610744715
loss at iteration 100: 0.5115529441833496
loss at iteration 120: 0.48986807043353714
loss at iteration 140: 0.47061881252697535
loss at iteration 160: 0.4532902978360653
loss at iteration 180: 0.43753159277968934
